# Sarva Conductor Train — Routely

**Repo private hai — GitHub link se mat kholo.**

## Pehle yeh 2 files upload karo (Colab left Files panel 📁)

1. `SARVA_CONDUCTOR_TRAIN.ipynb` — yeh notebook (File → Upload notebook)
2. `colab_export_full.zip` — PC path: `C:\Users\ujjwa\Saas\sarva_training\data\export\colab_export_full.zip`

Phir: **Runtime → A100** + Secret **`HF_TOKEN`** (Notebook access **ON**) → **Run all**

Trains 2154 rows → pushes `Ujjwal211/aitotech-sarva-v2`

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets huggingface_hub

In [ ]:
from huggingface_hub import login
from google.colab import userdata
import os

token = userdata.get('HF_TOKEN')
login(token=token)
os.environ['HF_TOKEN'] = token
print('HF login OK')

In [ ]:
import zipfile
import shutil
from pathlib import Path

DATA_PATH = Path('/content/conductor_v1_train.jsonl')

if DATA_PATH.exists():
    print('Dataset already ready:', DATA_PATH, 'bytes', DATA_PATH.stat().st_size)
else:
    ZIP_CANDIDATES = [Path('/content/colab_export_full.zip'), Path('/colab_export_full.zip')]
    ZIP_PATH = next((p for p in ZIP_CANDIDATES if p.exists()), None)
    if ZIP_PATH:
        print('Found zip:', ZIP_PATH)
        with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
            zf.extractall('/content/')
        for p in Path('/content').rglob('conductor_v1_train.jsonl'):
            if p.resolve() != DATA_PATH.resolve():
                shutil.copy(p, DATA_PATH)
            break
    if not DATA_PATH.exists():
        raise FileNotFoundError('Upload colab_export_full.zip or conductor_v1_train.jsonl')
    print('Dataset ready:', DATA_PATH, 'bytes', DATA_PATH.stat().st_size)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='nvidia/Nemotron-3-Nano-30B-A3B',
    max_seq_length=4096,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing=True,
)
print('Model + LoRA ready')

In [ ]:
import json
from datasets import Dataset

rows = []
with open('/content/conductor_v1_train.jsonl', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

def format_example(ex):
    return tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)

dataset = Dataset.from_list([{'text': format_example(r)} for r in rows])
print('Training rows:', len(dataset))

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir='/content/sarva_conductor_lora',
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=3,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.1,
        logging_steps=10,
        save_steps=200,
        fp16=True,
        dataset_text_field='text',
    ),
)
trainer.train()
print('Training complete')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/Aitotech/sarva_conductor_v2_lora'
!mkdir -p "$SAVE_DIR"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print('Saved to Drive:', SAVE_DIR)

In [ ]:
ADAPTER_REPO = 'Ujjwal211/aitotech-sarva-v2'
model.push_to_hub(ADAPTER_REPO)
tokenizer.push_to_hub(ADAPTER_REPO)
print('Pushed to', ADAPTER_REPO)